In [1]:
#!/usr/bin/env python3
"""
Geothermal Data Repository (GDR) Scraper
========================================

A single-file Python 3 web scraper that:
1. Iterates through a range of Submission IDs (default 1-2500).
2. Visits individual dataset pages.
3. Extracts metadata and the official BibTeX citation from the HTML.
4. Downloads associated files into organized directories.
5. Uses TinyDB to track progress and prevent re-downloading.

Usage from CLI:
    python gdr_scraper.py           # Start full crawl (IDs 1-2500)
    python gdr_scraper.py --id 1802 # Scrape only submission 1802
    python gdr_scraper.py --reset   # Clear DB and restart

Usage from Python:
    import gdr_scraper
    gdr_scraper.main(start_id=1, end_id=100)
"""

import os
import re
import json
import time
import logging
import argparse
import requests
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from tinydb import TinyDB, Query

# ------------------------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------------------------

BASE_URL = "https://gdr.openei.org"
# Range of IDs to scan if running a full crawl
DEFAULT_START_ID = 1688
DEFAULT_END_ID = 2600 

# Directory settings
OUTPUT_DIR = "gdr_data"
DB_FILENAME = "gdr_db.json"
FILES_DIR_NAME = "files"

# Network settings
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/91.0.4472.124 Safari/537.36"
)
TIMEOUT = 20  # Seconds
RETRY_COUNT = 3
RETRY_DELAY = 2  # Seconds
POLITENESS_DELAY = 0.5  # Seconds between requests

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("GDR_Scraper")
logger.setLevel(logging.INFO) 

# ------------------------------------------------------------------------------
# DATABASE WRAPPER
# ------------------------------------------------------------------------------

class DatabaseManager:
    """Manages TinyDB interactions for submissions and state tracking."""
    
    def __init__(self, db_path):
        self.db = TinyDB(db_path)
        self.table = self.db.table('submissions')
        self.Submission = Query()

    def exists(self, submission_id):
        """Check if a submission exists in the DB."""
        return self.table.contains(self.Submission.id == str(submission_id))

    def is_complete(self, submission_id):
        """Check if a submission is marked as complete."""
        result = self.table.get(self.Submission.id == str(submission_id))
        return result and result.get('status') == 'complete'

    def upsert_submission(self, data):
        """Insert or update a submission record."""
        # Ensure ID is stored as string for consistency
        data['id'] = str(data['id'])
        self.table.upsert(data, self.Submission.id == data['id'])

    def update_status(self, submission_id, status):
        """Update the processing status of a submission."""
        self.table.update({'status': status}, self.Submission.id == str(submission_id))

    def update_files(self, submission_id, file_list):
        """Update the file list for a submission."""
        self.table.update({'files': file_list}, self.Submission.id == str(submission_id))

    def clear(self):
        """Wipe the database."""
        self.db.drop_table('submissions')
        logger.warning("Database cleared.")

# ------------------------------------------------------------------------------
# NETWORK WRAPPER
# ------------------------------------------------------------------------------

def get_session():
    """Create a requests session with default headers."""
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})
    return session

def fetch_url(session, url, stream=False, allow_404=False):
    """
    Fetch a URL with retries. Returns None if 404 (and allow_404=True).
    """
    for attempt in range(1, RETRY_COUNT + 1):
        try:
            response = session.get(url, timeout=TIMEOUT, stream=stream)
            
            # Special handling for 404s (Dataset doesn't exist)
            if response.status_code == 404:
                if allow_404:
                    return None
                else:
                    logger.warning(f"404 Not Found: {url}")
                    return None
            
            response.raise_for_status()
            return response
        except requests.exceptions.RequestException as e:
            if attempt < RETRY_COUNT:
                time.sleep(RETRY_DELAY * attempt)
            else:
                logger.error(f"Failed to fetch {url} after {RETRY_COUNT} attempts: {e}")
                return None
    return None

def safe_filename(name):
    """Sanitize a string to be safe for filenames."""
    # Remove extension-like dots from the middle of the name to avoid confusion, 
    # keep only alphanumeric and basic separators
    clean = re.sub(r'[\\/*?:"<>|]', "", name).strip()
    return clean.replace(" ", "_")

# ------------------------------------------------------------------------------
# PARSING FUNCTIONS
# ------------------------------------------------------------------------------

def parse_dataset_page(html_content, submission_url, submission_id):
    """
    Extract metadata, bibtex, and file links from a dataset detail page.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    data = {
        'id': str(submission_id),
        'url': submission_url,
        'title': f"Submission {submission_id}",
        'description': "",
        'authors': [],
        'keywords': [],
        'date_published': "",
        'bibtex': None,
        'files': [], # List of dicts: {url, name, path, status}
        'metadata_source': 'html'
    }

    # -------------------------------------------------------------------------
    # 1. Extract BibTeX (Fixed Logic)
    # -------------------------------------------------------------------------
    # Target: <div id="bibtex"> ... <code> ... </code> </div>
    bibtex_div = soup.find('div', id='bibtex')
    if bibtex_div:
        code_block = bibtex_div.find('code')
        if code_block:
            data['bibtex'] = code_block.get_text().strip()
    
    # -------------------------------------------------------------------------
    # 2. Try JSON-LD (Schema.org) for metadata
    # -------------------------------------------------------------------------
    script_tag = soup.find('script', type='application/ld+json')
    if script_tag:
        try:
            json_data = json.loads(script_tag.string)
            data['metadata_source'] = 'json-ld'
            data['title'] = json_data.get('name', data['title'])
            data['description'] = json_data.get('description', "")
            data['date_published'] = json_data.get('datePublished', "")
            
            # Authors
            creators = json_data.get('creator', [])
            if isinstance(creators, dict): creators = [creators]
            data['authors'] = [c.get('name') for c in creators if 'name' in c]
            
            # Keywords
            data['keywords'] = json_data.get('keywords', [])

            # Files (Distribution)
            distributions = json_data.get('distribution', [])
            if isinstance(distributions, dict): distributions = [distributions]
            
            for dist in distributions:
                file_url = dist.get('contentUrl')
                if file_url:
                    # Guess filename from URL
                    fname = os.path.basename(urlparse(file_url).path) or "unknown_file"
                    data['files'].append({
                        'url': file_url,
                        'name': fname,
                        'description': dist.get('about') or dist.get('description', ''),
                        'status': 'pending'
                    })
                    
        except json.JSONDecodeError:
            logger.warning(f"Failed to parse JSON-LD for {submission_id}, falling back to HTML.")

    # -------------------------------------------------------------------------
    # 3. HTML Fallback for basic data
    # -------------------------------------------------------------------------
    if data['title'] == f"Submission {submission_id}":
        header = soup.find('h1') or soup.find('h2', class_='page-header')
        if header:
            data['title'] = header.get_text(strip=True)

    # 4. HTML Fallback for files
    if not data['files']:
        # Look for links containing /files/ID/
        file_links = soup.find_all('a', href=re.compile(rf'/files/{submission_id}/'))
        for link in file_links:
            href = link['href']
            full_url = urljoin(BASE_URL, href)
            
            # Avoid duplicate links
            if any(f['url'] == full_url for f in data['files']):
                continue
                
            fname = os.path.basename(urlparse(full_url).path)
            data['files'].append({
                'url': full_url,
                'name': fname,
                'description': link.get_text(strip=True),
                'status': 'pending'
            })

    return data

# ------------------------------------------------------------------------------
# DOWNLOADER
# ------------------------------------------------------------------------------

def download_files(session, submission_data, base_download_dir):
    """
    Download all pending files for a submission.
    Updates the 'files' list in submission_data with local paths and status.
    """
    sub_id = submission_data['id']
    # Create folder: downloads/1234_Safe_Title/
    safe_title = safe_filename(submission_data['title'])[:50]
    folder_name = f"{sub_id}_{safe_title}"
    target_dir = base_download_dir / folder_name
    target_dir.mkdir(parents=True, exist_ok=True)
    
    # Save BibTeX file if found
    if submission_data.get('bibtex'):
        bib_path = target_dir / "citation.bib"
        try:
            with open(bib_path, 'w', encoding='utf-8') as f:
                f.write(submission_data['bibtex'])
            logger.info("  BibTeX citation saved.")
        except Exception as e:
            logger.error(f"  Failed to save BibTeX: {e}")

    for file_info in submission_data['files']:
        if file_info.get('status') == 'downloaded':
            continue

        url = file_info['url']
        filename = safe_filename(file_info['name'])
        file_path = target_dir / filename
        
        logger.info(f"  Downloading: {filename}")
        
        response = fetch_url(session, url, stream=True)
        if response:
            try:
                with open(file_path, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                
                file_info['local_path'] = str(file_path)
                file_info['status'] = 'downloaded'
            except Exception as e:
                logger.error(f"  Error writing file {filename}: {e}")
                file_info['status'] = 'failed'
                file_info['error'] = str(e)
        else:
            file_info['status'] = 'failed_network'

    return submission_data

# ------------------------------------------------------------------------------
# MAIN EXECUTION LOGIC
# ------------------------------------------------------------------------------

def process_submission(session, db, submission_id, url, download_dir, skip_downloads=False):
    """
    Full workflow for a single submission: Fetch -> Parse -> Save -> Download.
    """
    submission_id = str(submission_id)
    
    # 1. Check DB
    if db.is_complete(submission_id):
        logger.info(f"Skipping {submission_id} (Already complete).")
        return

    # 2. Fetch Page (Handle 404s gracefully)
    response = fetch_url(session, url, allow_404=True)
    if not response:
        # If response is None, it means 404 (doesn't exist) or network failure
        logger.info(f"Submission {submission_id} does not exist (404) or connection failed.")
        return

    #breakpoint()
    logger.info(f"Processing Submission {submission_id}...")

    # 3. Parse Metadata
    try:
        data = parse_dataset_page(response.text, url, submission_id)
    except Exception as e:
        logger.error(f"Error parsing page {submission_id}: {e}")
        return

    # Save initial metadata
    data['status'] = 'metadata_extracted'
    db.upsert_submission(data)

    # 4. Download Files
    if not skip_downloads:
        if data['files']:
            logger.info(f"  Found {len(data['files'])} files. Starting downloads...")
            data = download_files(session, data, download_dir)
            db.update_files(submission_id, data['files'])
        else:
            logger.info("  No files found to download.")

    # 5. Mark Complete
    db.update_status(submission_id, 'complete')
    logger.info(f"Finished {submission_id}.")
    
    # Be polite
    time.sleep(POLITENESS_DELAY)

def main(submission_id=None, start_id=DEFAULT_START_ID, end_id=DEFAULT_END_ID, 
         reset=False, no_download=False, output_dir=OUTPUT_DIR):
    """
    Main programmatic entry point.
    
    :param submission_id: (Optional) ID of a specific submission to scrape.
    :param start_id: Starting ID for iteration (default 1).
    :param end_id: Ending ID for iteration (default 2500).
    :param reset: If True, clear the DB before starting.
    :param no_download: If True, skip file downloads (metadata only).
    :param output_dir: Directory to save DB and files.
    """
    
    # Setup Paths
    root_dir = Path(output_dir)
    root_dir.mkdir(parents=True, exist_ok=True)
    db_path = root_dir / DB_FILENAME
    files_dir = root_dir / FILES_DIR_NAME
    
    #breakpoint()
    # Initialize DB
    db = DatabaseManager(str(db_path))
    if reset:
        db.clear()

    session = get_session()

    # Case A: Scrape a single ID
    if submission_id:
        url = f"{BASE_URL}/submissions/{submission_id}"
        process_submission(session, db, submission_id, url, files_dir, no_download)
        return

    # Case B: Iterate through a range of IDs
    logger.info(f"Starting iteration from ID {start_id} to {end_id}...")
    
    for current_id in range(start_id, end_id + 1):
        url = f"{BASE_URL}/submissions/{current_id}"
        process_submission(session, db, current_id, url, files_dir, no_download)
    
    logger.info("Iteration complete.")

def cli_entry_point():
    """Parses CLI arguments and calls main()."""
    parser = argparse.ArgumentParser(description="Geothermal Data Repository Scraper")
    
    parser.add_argument("--id", type=str, help="Scrape a specific Submission ID only")
    parser.add_argument("--start", type=int, default=DEFAULT_START_ID, help=f"Start ID (default {DEFAULT_START_ID})")
    parser.add_argument("--end", type=int, default=DEFAULT_END_ID, help=f"End ID (default {DEFAULT_END_ID})")
    parser.add_argument("--reset", action="store_true", help="Clear database and start fresh")
    parser.add_argument("--no-download", action="store_true", help="Scrape metadata only, skip files")
    parser.add_argument("--out", type=str, default=OUTPUT_DIR, help="Base directory for downloads")
    
    args = parser.parse_args()
    
    main(
        submission_id=args.id,
        start_id=args.start,
        end_id=args.end,
        reset=args.reset,
        no_download=args.no_download,
        output_dir=args.out
    )

if __name__ == "__main__":
    main()

20:21:49 [INFO] Starting iteration from ID 1688 to 2600...
20:21:49 [INFO] Skipping 1688 (Already complete).
20:21:49 [INFO] Skipping 1689 (Already complete).
20:21:49 [INFO] Skipping 1690 (Already complete).
20:21:49 [INFO] Skipping 1691 (Already complete).
20:21:49 [INFO] Skipping 1692 (Already complete).
20:21:49 [INFO] Skipping 1693 (Already complete).
20:21:49 [INFO] Skipping 1694 (Already complete).
20:21:49 [INFO] Skipping 1695 (Already complete).
20:21:49 [INFO] Skipping 1696 (Already complete).
20:21:50 [INFO] Skipping 1697 (Already complete).
20:21:50 [INFO] Skipping 1698 (Already complete).
20:21:50 [INFO] Skipping 1699 (Already complete).
20:21:50 [INFO] Skipping 1700 (Already complete).
20:21:50 [INFO] Skipping 1701 (Already complete).
20:21:50 [INFO] Skipping 1702 (Already complete).
20:21:50 [INFO] Skipping 1703 (Already complete).
20:21:50 [INFO] Skipping 1704 (Already complete).
20:21:50 [INFO] Skipping 1705 (Already complete).
20:21:50 [INFO] Skipping 1706 (Already co